<a href="https://colab.research.google.com/github/andrellyrichelly-hub/ATIVIDADE-PROFESSOR-CLAUDOMIRO_-ALUNA-DE-IA-ANDRELLY-UFPA/blob/main/ATIVI_2_6_3_ANDRELLY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Atividade 2.6 - Slide 58: Exemplos 5.8, 5.9 e 5.10
# Simulador de Cache - Hit/Miss
# Andrelly Richelly - Arquitetura com Claudomiro

class LinhaCache:
    def __init__(self):
        self.valida = 0
        self.tag = -1
        self.dados = None
        self.lru = 0 # pra substituição

class CacheSimulador:
    """
    MP=4GB, Bloco=64B, Cache=64KB, 1024 Linhas
    5.8: Direto | 5.9: Total Assoc | 5.10: Conj 32-way
    """
    def __init__(self, tipo):
        self.tipo = tipo
        self.num_linhas = 1024
        self.hits = 0
        self.misses = 0

        if tipo == "DIRETO": # 5.8
            self.num_conjuntos = 1024
            self.linhas_por_conj = 1
            self.bits_conj = 10
        elif tipo == "TOTAL_ASSOCIATIVO": # 5.9
            self.num_conjuntos = 1
            self.linhas_por_conj = 1024
            self.bits_conj = 0
        elif tipo == "CONJ_32WAY": # 5.10
            self.num_conjuntos = 32
            self.linhas_por_conj = 32
            self.bits_conj = 5

        # Inicializa cache
        self.cache = [[LinhaCache() for _ in range(self.linhas_por_conj)]
                      for _ in range(self.num_conjuntos)]

    def acessar(self, endereco):
        offset = endereco & 0x3F
        bloco = endereco >> 6

        if self.tipo == "DIRETO":
            linha = bloco & 0x3FF
            tag = bloco >> 10
            conjunto = linha
            via = 0
        elif self.tipo == "TOTAL_ASSOCIATIVO":
            tag = bloco
            conjunto = 0
            via = -1 # busca em todas
        else: # CONJ_32WAY
            conjunto = bloco & 0x1F
            tag = bloco >> 5
            via = -1 # busca nas 32 vias

        # Verifica hit
        hit = False
        via_hit = -1
        for i in range(self.linhas_por_conj):
            linha_atual = self.cache[conjunto][i]
            if linha_atual.valida and linha_atual.tag == tag:
                hit = True
                via_hit = i
                linha_atual.lru = 0 # acessou, zera LRU
                break

        # Atualiza LRU das outras linhas do conjunto
        for i in range(self.linhas_por_conj):
            if i!= via_hit:
                self.cache[conjunto][i].lru += 1

        if hit:
            self.hits += 1
            status = "HIT"
        else:
            self.misses += 1
            status = "MISS"
            # Substituição: pega a linha com maior LRU
            via_subst = max(range(self.linhas_por_conj),
                           key=lambda i: self.cache[conjunto][i].lru)
            self.cache[conjunto][via_subst].valida = 1
            self.cache[conjunto][via_subst].tag = tag
            self.cache[conjunto][via_subst].lru = 0
            via_hit = via_subst

        print(f"{self.tipo:<18} | End: 0x{endereco:08X} | Tag: {tag:05d} | Conj/Linha: {conjunto:04d} | Via: {via_hit:02d} | {status}")
        return hit

# --- TESTE AUTOMÁTICO - EXEMPLOS 5.8, 5.9 e 5.10 ---
print("=== ATIVIDADE 2.6 - SLIDE 58: EXEMPLOS 5.8, 5.9 e 5.10 ===\n")

# Sequência de endereços que força conflito no Direto
sequencia = [0x00000000, 0x00010000, 0x00000040, 0x00010040, 0x00000000]

for tipo in ["DIRETO", "TOTAL_ASSOCIATIVO", "CONJ_32WAY"]:
    print(f"\n--- 5.{8 if tipo=='DIRETO' else 9 if tipo=='TOTAL_ASSOCIATIVO' else 10} MAPEAMENTO {tipo} ---")
    cache = CacheSimulador(tipo)
    for end in sequencia:
        cache.acessar(end)
    total = cache.hits + cache.misses
    taxa_hit = cache.hits / total * 100
    print(f"RESULTADO: Hits={cache.hits} | Misses={cache.misses} | Taxa Hit={taxa_hit:.1f}%")
    print("-" * 90)

print("\nTeste concluído com sucesso.")

=== ATIVIDADE 2.6 - SLIDE 58: EXEMPLOS 5.8, 5.9 e 5.10 ===


--- 5.8 MAPEAMENTO DIRETO ---
DIRETO             | End: 0x00000000 | Tag: 00000 | Conj/Linha: 0000 | Via: 00 | MISS
DIRETO             | End: 0x00010000 | Tag: 00001 | Conj/Linha: 0000 | Via: 00 | MISS
DIRETO             | End: 0x00000040 | Tag: 00000 | Conj/Linha: 0001 | Via: 00 | MISS
DIRETO             | End: 0x00010040 | Tag: 00001 | Conj/Linha: 0001 | Via: 00 | MISS
DIRETO             | End: 0x00000000 | Tag: 00000 | Conj/Linha: 0000 | Via: 00 | MISS
RESULTADO: Hits=0 | Misses=5 | Taxa Hit=0.0%
------------------------------------------------------------------------------------------

--- 5.9 MAPEAMENTO TOTAL_ASSOCIATIVO ---
TOTAL_ASSOCIATIVO  | End: 0x00000000 | Tag: 00000 | Conj/Linha: 0000 | Via: 00 | MISS
TOTAL_ASSOCIATIVO  | End: 0x00010000 | Tag: 01024 | Conj/Linha: 0000 | Via: 01 | MISS
TOTAL_ASSOCIATIVO  | End: 0x00000040 | Tag: 00001 | Conj/Linha: 0000 | Via: 02 | MISS
TOTAL_ASSOCIATIVO  | End: 0x00010040 | Tag: